In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [2]:
BASE_URL = "https://api.deepseek.com"
DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
deepseek_chat_model = "deepseek-chat"
deepseek_reason_model = "deepseek-reasoner"

## Chat model

In [3]:
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import AIMessage, HumanMessage

In [4]:
model = ChatOpenAI(base_url=BASE_URL, api_key=DEEPSEEK_API_KEY, model=deepseek_chat_model)
messages = [HumanMessage("Who are you?")]
model.invoke(messages)

AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n\n我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。\n\n你可以通过官方应用商店下载我的App来使用。我很乐意帮助你解答问题、处理文档、进行对话交流等等！\n\n有什么我可以帮助你的吗？无论是学习、工作还是日常问题，我都很愿意为你提供帮助！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 8, 'total_tokens': 148, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '65f3f200-5e22-40cd-8b34-7170b4ce2953', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--019c7df6-fab4-7552-ad18-c40af19f8317-0', usage_metadata={'input_tokens': 8, 'output_tokens': 140, 'total_tokens': 148, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

# Prompt template

In [5]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate(
   [
    ("ai", "You are a helpful assistant."),
    ("human", "Help me write a joke about {topic} in {language}")
   ]
)
prompt_template.input_variables

['language', 'topic']

In [6]:
prompt_template.invoke({"topic": "码头", "language": "English"}).to_messages()

[AIMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Help me write a joke about 码头 in English', additional_kwargs={}, response_metadata={})]

In [7]:
chain = prompt_template | model
chain.invoke({"topic": "码头", "language": "English"}).content

'Why did the dockworker bring a ladder to the码头?\n\nBecause he heard the work was on another level!'

# Structured output

In [8]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

In [9]:
class Joke(BaseModel):
    "Joke to tell user."
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")

model_dpsk = ChatDeepSeek(api_key=DEEPSEEK_API_KEY, model=deepseek_chat_model)
joke_chain = prompt_template | model_dpsk.with_structured_output(Joke)
joke_chain.invoke({"topic": "码头", "language": "English"})

Joke(setup='Why did the cargo ship refuse to leave the dock?', punchline='Because it had too many pier pressure!')

# Memory

In [10]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

In [11]:
work_flow = StateGraph(state_schema=MessagesState)

def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": response}

In [12]:
work_flow.add_node("model", call_model)
work_flow.add_edge(START, "model")

In [13]:
memory = MemorySaver()
graph = work_flow.compile(checkpointer=memory)

In [14]:
query = "Hi, I'm Alvin."
config = {"configurable": {"thread_id": "abc123"}}
input_messages = [HumanMessage(query)]
output = graph.invoke({"messages": input_messages}, config)

In [15]:
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Nice to meet you, Alvin! I'm DeepSeek, an AI assistant created by DeepSeek Company. I'm here to help you with questions, conversations, or any tasks you might need assistance with. 

Is there anything specific you'd like to talk about or ask today? Whether it's about learning something new, solving a problem, or just having a friendly chat, I'm here to help! 😊


In [16]:
query = "What is my name?"
input_messages = [HumanMessage(query)]
output = graph.invoke({"messages": input_messages}, config)

In [17]:
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

You told me your name is **Alvin**! 😊  
Is there anything else you'd like to talk about, Alvin?


In [18]:
query = "What is my name?"
config = {"configurable": {"thread_id": "abc345"}}
input_messages = [HumanMessage(query)]
output = graph.invoke({"messages": input_messages}, config)

In [19]:
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

I don’t have access to your personal information, so I don’t know your name. 😊  
If you’d like, you can tell me your name, and I’ll use it in our conversation — or we can keep things anonymous.


# Agent

In [20]:
from langgraph.prebuilt import create_react_agent

In [21]:
def add(a: int, b: int) -> int:
    "Add two integers."
    return a + b

def multiply(a: int, b: int) -> int:
    "Multiply two integers."
    return a * b

In [ ]:
agent = create_react_agent(model=model_dpsk,tools=[add, multiply])

In [23]:
output = agent.invoke({"messages": [HumanMessage("Compute 99 + 98 * 97")]})

In [24]:
for message in output["messages"]:
    message.pretty_print()

================================ Human Message =================================

Compute 99 + 98 * 97
================================== Ai Message ==================================

I'll help you compute 99 + 98 * 97. According to the order of operations (PEMDAS/BODMAS), multiplication comes before addition, so I need to compute 98 * 97 first, then add 99 to that result.

Let me start with the multiplication:
Tool Calls:
  multiply (call_00_e2PmBdkg9aTf9xLC1OnKxIdb)
 Call ID: call_00_e2PmBdkg9aTf9xLC1OnKxIdb
  Args:
    a: 98
    b: 97
================================= Tool Message =================================
Name: multiply

9506
================================== Ai Message ==================================

Now I need to add 99 to this result:
Tool Calls:
  add (call_00_hjuxmKY8XJyi3CaUvGszVJsH)
 Call ID: call_00_hjuxmKY8XJyi3CaUvGszVJsH
  Args:
    a: 99
    b: 9506
================================= Tool Message =================================
Name: add

9605
===========

# Deep Research

## Plan agent、Search agent、Report agent

In [25]:
from langchain_tavily import TavilySearch